# The Diffusion Paradigm (Colab)

Companion notebook for the [The Diffusion Paradigm](https://overfitting.club/posts/tutorials/deep_learning/the_diffusion_paradigm/the_diffusion_paradigm.html) tutorial. We implement the full **DDPM pipeline** — forward noising, noise schedules, the training objective, and reverse sampling — using a lightweight class-conditional U-Net on FashionMNIST.

See the full tutorial for theory, derivations, and architecture diagrams.

**What you'll build:**
- A forward noising process with linear and cosine schedules
- A class-conditional U-Net denoiser with timestep + class embeddings
- The DDPM training loop with EMA
- Interactive widgets for generation, denoising visualization, and schedule exploration

In [ ]:
#@title Install dependencies
try:
    import torch, plotly, rich, tensorboard
except ImportError:
    !pip install -q torch torchvision plotly ipywidgets rich tensorboard

### Hyperparameters

| Parameter | What it controls |
|-----------|------------------|
| `T` | Total diffusion steps (1000 in DDPM) |
| `BATCH_SIZE` | Training batch size |
| `EPOCHS` | Number of training epochs |
| `LR` | Adam learning rate |
| `EMA_DECAY` | Exponential moving average decay |
| `DROPOUT` | Dropout rate in ResBlocks |

In [ ]:
#@title Configuration { run: "auto" }

T = 1000  #@param {type:"integer"}
BATCH_SIZE = 256  #@param {type:"integer"}
EPOCHS = 200  #@param {type:"integer"}
LR = 2e-4  #@param {type:"number"}
EMA_DECAY = 0.999  #@param {type:"number"}
GRAD_CLIP_NORM = 1.0  #@param {type:"number"}
DROPOUT = 0.1  #@param {type:"number"}
NUM_CLASSES = 10

from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torch.utils.tensorboard import SummaryWriter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from rich.console import Console
from rich.table import Table

console = Console()
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

CHECKPOINT_DIR = Path("data/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

writer = SummaryWriter("runs/cddpm")

console.print(f"Device: {DEVICE} | T = {T}")

## 1. Dataset: FashionMNIST

Same dataset as previous tutorials — 28×28 grayscale images, normalized to [-1, 1].

In [ ]:
#@title Load FashionMNIST

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_data = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST("data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)
console.print(f"Train: {len(train_data):,} | Test: {len(test_data):,}")

## 2. The Forward Process — Destroying Information

The forward process adds Gaussian noise progressively until the image becomes pure noise. Using the closed-form formula, we can jump to any timestep in one shot:

$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\,\boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

In [ ]:
#@title Linear noise schedule and q_sample

def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    """DDPM linear schedule: β_t from beta_start to beta_end."""
    return torch.linspace(beta_start, beta_end, T)


betas = linear_beta_schedule(T)
alphas = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)


def q_sample(x_0, t, noise=None):
    """
    Forward process: sample x_t given x_0 using the closed-form formula.
    x_0:   (B, 1, H, W) clean images in [-1, 1]
    t:     (B,) integer timesteps in [0, T-1]
    noise: optional pre-sampled ε ~ N(0, I), same shape as x_0
    """
    if noise is None:
        noise = torch.randn_like(x_0)
    sqrt_alpha_bar = alpha_bar[t].sqrt()[:, None, None, None]
    sqrt_one_minus = (1.0 - alpha_bar[t]).sqrt()[:, None, None, None]
    return sqrt_alpha_bar * x_0 + sqrt_one_minus * noise

In [ ]:
#@title Mean and variance of q(x_t | x_0) over time

_betas = torch.linspace(1e-4, 0.02, T).numpy()
_alpha_bar = torch.cumprod(1.0 - torch.tensor(_betas), dim=0).numpy()

t_range = np.arange(T)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=t_range, y=np.sqrt(_alpha_bar),
    name="\u221a\u0101\u0332_t (mean coefficient)", line=dict(color="#4A90D9", width=2),
))
fig.add_trace(go.Scatter(
    x=t_range, y=1.0 - _alpha_bar,
    name="1 \u2212 \u0101\u0332_t (variance)", line=dict(color="#E74C3C", width=2),
))
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5)
fig.add_hline(y=1, line_dash="dot", line_color="gray", opacity=0.5)
fig.update_layout(
    title="Forward process: mean and variance of q(x_t | x_0)",
    xaxis_title="Timestep t",
    yaxis_title="Value",
    height=320, width=700,
    template="plotly_white",
    legend=dict(x=0.45, y=0.5),
    margin=dict(t=50, b=50, l=60, r=20),
)
fig.show()

In [ ]:
#@title Visualize forward noising at selected timesteps

sample_img, sample_label = test_data[0]
x_0 = sample_img.unsqueeze(0)

timesteps = [0, 50, 100, 200, 500, 999]

fig = make_subplots(
    rows=1, cols=len(timesteps),
    subplot_titles=[f"t = {t}" for t in timesteps],
    horizontal_spacing=0.03,
)

for i, t in enumerate(timesteps):
    t_tensor = torch.tensor([t])
    x_t = q_sample(x_0, t_tensor).squeeze().numpy()
    fig.add_trace(
        go.Heatmap(
            z=x_t[::-1], colorscale="Gray_r", showscale=False,
            hovertemplate="pixel (%{x}, %{y}): %{z:.2f}<extra></extra>",
        ),
        row=1, col=i + 1,
    )
    fig.update_xaxes(showticklabels=False, row=1, col=i + 1)
    fig.update_yaxes(showticklabels=False, row=1, col=i + 1)

fig.update_layout(
    title_text=f"Forward noising: {CLASS_NAMES[sample_label]} \u2192 pure noise",
    height=220, width=900,
    margin=dict(t=60, b=10, l=10, r=10),
    template="plotly_white",
)
fig.show()

## 3. Noise Schedules — Linear vs Cosine

The **linear schedule** increases $\beta_t$ linearly from $10^{-4}$ to $0.02$. The **cosine schedule** (Nichol & Dhariwal, 2021) defines $\bar{\alpha}_t$ directly through a cosine function, distributing the signal-to-noise ratio more uniformly.

We use the **cosine schedule** for the rest of this notebook.

In [ ]:
#@title Cosine schedule and comparison

def cosine_beta_schedule(T, s=0.008):
    """Nichol & Dhariwal cosine schedule."""
    steps = torch.arange(T + 1, dtype=torch.float64)
    f = torch.cos((steps / T + s) / (1 + s) * torch.pi / 2) ** 2
    alpha_bar_cos = f / f[0]
    betas_cos = 1 - (alpha_bar_cos[1:] / alpha_bar_cos[:-1])
    return betas_cos.clamp(max=0.999).float()


betas_linear = linear_beta_schedule(T)
alpha_bar_linear = torch.cumprod(1.0 - betas_linear, dim=0)

betas_cosine = cosine_beta_schedule(T)
alpha_bar_cosine = torch.cumprod(1.0 - betas_cosine, dim=0)

# Switch to cosine for the rest of the notebook
betas = betas_cosine
alphas = 1.0 - betas
alpha_bar = alpha_bar_cosine

In [ ]:
#@title \u0101\u0332_t comparison: linear vs cosine

t_range = np.arange(T)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=t_range, y=alpha_bar_linear.numpy(),
    name="Linear", line=dict(color="#E74C3C", width=2),
))
fig.add_trace(go.Scatter(
    x=t_range, y=alpha_bar_cosine.numpy(),
    name="Cosine", line=dict(color="#4A90D9", width=2),
))
fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5)
fig.add_hline(y=1, line_dash="dot", line_color="gray", opacity=0.5)
fig.update_layout(
    title="Cumulative signal retention: linear vs cosine",
    xaxis_title="Timestep t",
    yaxis_title="\u0101\u0332_t",
    height=320, width=700,
    template="plotly_white",
    legend=dict(x=0.65, y=0.9),
    margin=dict(t=50, b=50, l=60, r=20),
)
fig.show()

In [ ]:
#@title Noising comparison: linear vs cosine (same image, same noise)

timesteps_cmp = [0, 100, 250, 500, 750, 999]
schedules = [
    ("Linear", alpha_bar_linear),
    ("Cosine", alpha_bar_cosine),
]

fig = make_subplots(
    rows=2, cols=len(timesteps_cmp),
    subplot_titles=[f"t = {t}" for t in timesteps_cmp],
    row_titles=["Linear", "Cosine"],
    horizontal_spacing=0.03,
    vertical_spacing=0.08,
)

x_0_single = sample_img.unsqueeze(0)
torch.manual_seed(42)

for row_idx, (sched_name, abar) in enumerate(schedules):
    for col_idx, t in enumerate(timesteps_cmp):
        torch.manual_seed(42)
        noise = torch.randn_like(x_0_single)
        sqrt_abar = abar[t].sqrt()
        sqrt_one_minus = (1.0 - abar[t]).sqrt()
        x_t = sqrt_abar * x_0_single + sqrt_one_minus * noise
        fig.add_trace(
            go.Heatmap(
                z=x_t.squeeze().numpy()[::-1], colorscale="Gray_r", showscale=False,
                hovertemplate="pixel (%{x}, %{y}): %{z:.2f}<extra></extra>",
            ),
            row=row_idx + 1, col=col_idx + 1,
        )
        fig.update_xaxes(showticklabels=False, row=row_idx + 1, col=col_idx + 1)
        fig.update_yaxes(showticklabels=False, row=row_idx + 1, col=col_idx + 1)

fig.update_layout(
    title_text="Same image, same noise \u2014 linear vs cosine schedule",
    height=380, width=900,
    margin=dict(t=60, b=10, l=80, r=10),
    template="plotly_white",
)
fig.show()

## 4. Architecture — Class-Conditional U-Net

The denoiser takes $(\mathbf{x}_t, t, y)$ as input and predicts $\hat{\boldsymbol{\varepsilon}}$. Key components:
- **Sinusoidal timestep embedding** — encodes *how noisy* the input is
- **Class embedding** — encodes *what* we are denoising
- **ResBlocks with embedding injection** — time + class info at every layer
- **Self-attention at 14×14** — lets distant pixels coordinate
- **Skip connections** — standard U-Net encoder-decoder pattern

In [ ]:
#@title Sinusoidal timestep embedding

class SinusoidalPE(nn.Module):
    """Sinusoidal positional encoding for integer timesteps."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half = self.dim // 2
        freqs = torch.exp(
            -np.log(10000) * torch.arange(half, device=device) / half
        )
        args = t[:, None].float() * freqs[None, :]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class TimestepEmbedding(nn.Module):
    """SinusoidalPE \u2192 MLP \u2192 embedding vector."""
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or 4 * dim
        self.pe = SinusoidalPE(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t):
        return self.mlp(self.pe(t))

In [ ]:
#@title Building blocks: ResBlock, AttnBlock, up/down

class ResBlock(nn.Module):
    """GroupNorm \u2192 SiLU \u2192 Conv, twice, with timestep+class embedding injection."""
    def __init__(self, in_ch, out_ch, emb_dim, dropout=DROPOUT):
        super().__init__()
        self.norm1 = nn.GroupNorm(32, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.norm2 = nn.GroupNorm(32, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.skip(x)


class AttnBlock(nn.Module):
    """Single-head self-attention at a fixed spatial resolution."""
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(32, ch)
        self.qkv  = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        q, k, v = self.qkv(self.norm(x)).chunk(3, dim=1)
        q = q.reshape(B, C, H * W).permute(0, 2, 1)
        k = k.reshape(B, C, H * W)
        v = v.reshape(B, C, H * W).permute(0, 2, 1)
        a = torch.bmm(q, k).mul_(C ** -0.5).softmax(dim=-1)
        h = torch.bmm(a, v).permute(0, 2, 1).reshape(B, C, H, W)
        return x + self.proj(h)


def downsample(ch):
    return nn.Conv2d(ch, ch, 3, stride=2, padding=1)

def upsample(ch):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode="nearest"),
        nn.Conv2d(ch, ch, 3, padding=1),
    )

In [ ]:
#@title Class-conditional U-Net

class CondUNet(nn.Module):
    """
    Class-conditional U-Net for 1\u00d728\u00d728 images.

    Encoder: 1 \u2192 64 \u2192 128 \u2192 128           (spatial: 28 \u2192 14 \u2192 7)
    Middle:  ResBlock \u2192 AttnBlock \u2192 ResBlock  (128, 7\u00d77)
    Decoder: 128 \u2192 128 \u2192 64 \u2192 1           (spatial: 7 \u2192 14 \u2192 28)
    """
    def __init__(
        self, in_ch=1, ch=64, num_classes=NUM_CLASSES,
        ch_mult=(1, 2, 2), num_res_blocks=2, attn_resolutions=(14,),
    ):
        super().__init__()
        emb_dim = ch * 4
        self.time_emb = TimestepEmbedding(ch, emb_dim)
        self.class_emb = nn.Embedding(num_classes, emb_dim)

        self.entry = nn.Conv2d(in_ch, ch, 3, padding=1)

        # --- Encoder ---------------------------------------------------------
        self.down_layers = nn.ModuleList()
        skip_channels = [ch]
        cur_ch, cur_res = ch, 28
        for i, mult in enumerate(ch_mult):
            out_ch = ch * mult
            for _ in range(num_res_blocks):
                block = nn.ModuleDict({"res": ResBlock(cur_ch, out_ch, emb_dim)})
                if cur_res in attn_resolutions:
                    block["attn"] = AttnBlock(out_ch)
                self.down_layers.append(block)
                cur_ch = out_ch
                skip_channels.append(cur_ch)
            if i < len(ch_mult) - 1:
                self.down_layers.append(nn.ModuleDict({"down": downsample(cur_ch)}))
                skip_channels.append(cur_ch)
                cur_res //= 2

        # --- Middle ----------------------------------------------------------
        self.mid_res1 = ResBlock(cur_ch, cur_ch, emb_dim)
        self.mid_attn = AttnBlock(cur_ch)
        self.mid_res2 = ResBlock(cur_ch, cur_ch, emb_dim)

        # --- Decoder ---------------------------------------------------------
        self.up_layers = nn.ModuleList()
        for i, mult in enumerate(reversed(ch_mult)):
            out_ch = ch * mult
            for _ in range(num_res_blocks + 1):
                skip_ch = skip_channels.pop()
                block = nn.ModuleDict({"res": ResBlock(cur_ch + skip_ch, out_ch, emb_dim)})
                if cur_res in attn_resolutions:
                    block["attn"] = AttnBlock(out_ch)
                self.up_layers.append(block)
                cur_ch = out_ch
            if i < len(ch_mult) - 1:
                self.up_layers.append(nn.ModuleDict({"up": upsample(cur_ch)}))
                cur_res *= 2

        self.out_norm = nn.GroupNorm(32, cur_ch)
        self.out = nn.Conv2d(cur_ch, in_ch, 3, padding=1)

    def forward(self, x, t, y):
        emb = self.time_emb(t) + self.class_emb(y)

        h = self.entry(x)
        hs = [h]
        for block in self.down_layers:
            if "down" in block:
                h = block["down"](h)
            else:
                h = block["res"](h, emb)
                if "attn" in block:
                    h = block["attn"](h)
            hs.append(h)

        h = self.mid_res1(h, emb)
        h = self.mid_attn(h)
        h = self.mid_res2(h, emb)

        for block in self.up_layers:
            if "up" in block:
                h = block["up"](h)
            else:
                skip = hs.pop()
                if h.shape[-2:] != skip.shape[-2:]:
                    h = F.pad(h, [0, skip.shape[-1] - h.shape[-1],
                                  0, skip.shape[-2] - h.shape[-2]])
                h = torch.cat([h, skip], dim=1)
                h = block["res"](h, emb)
                if "attn" in block:
                    h = block["attn"](h)

        return self.out(F.silu(self.out_norm(h)))

In [ ]:
#@title Model summary

model = CondUNet(in_ch=1, ch=64).to(DEVICE)

with torch.no_grad():
    dummy_x = torch.randn(2, 1, 28, 28, device=DEVICE)
    dummy_t = torch.randint(0, T, (2,), device=DEVICE)
    dummy_y = torch.randint(0, NUM_CLASSES, (2,), device=DEVICE)
    dummy_out = model(dummy_x, dummy_t, dummy_y)

tbl = Table(title="CondUNet Architecture")
tbl.add_column("Stage", style="cyan")
tbl.add_column("Layers", style="magenta")
tbl.add_column("Channels", style="green")
tbl.add_column("Spatial", style="dim")
for stage, layers, ch_str, spatial in [
    ("Input",   "\u2014",                             "1",   "28\u00d728"),
    ("Entry",   "Conv 3\u00d73",                      "64",  "28\u00d728"),
    ("Level 1", "2\u00d7 ResBlock",                   "64",  "28\u00d728"),
    ("Down 1",  "Conv \u21932",                       "64",  "14\u00d714"),
    ("Level 2", "2\u00d7 (ResBlock + Attn)",          "128", "14\u00d714"),
    ("Down 2",  "Conv \u21932",                       "128", "7\u00d77"),
    ("Level 3", "2\u00d7 ResBlock",                   "128", "7\u00d77"),
    ("Middle",  "ResBlock + Attn + ResBlock",    "128", "7\u00d77"),
    ("Up  3",   "3\u00d7 ResBlock",                   "128", "7\u00d77"),
    ("Up  \u21912",  "NN\u21912 + Conv",              "128", "14\u00d714"),
    ("Up  2",   "3\u00d7 (ResBlock + Attn)",          "128", "14\u00d714"),
    ("Up  \u21912",  "NN\u21912 + Conv",              "128", "28\u00d728"),
    ("Up  1",   "3\u00d7 ResBlock",                   "64",  "28\u00d728"),
    ("Output",  "GroupNorm + SiLU + Conv 3\u00d73",   "1",   "28\u00d728"),
]:
    tbl.add_row(stage, layers, ch_str, spatial)
console.print(tbl)

total = sum(p.numel() for p in model.parameters())
console.print(f"\n[bold]Total parameters:[/bold] {total:,}")
console.print(f"[bold]Input \u2192 Output:[/bold] {list(dummy_x.shape)} \u2192 {list(dummy_out.shape)}")

## 5. Training

The DDPM training algorithm:
1. Sample a clean image $\mathbf{x}_0$ from the dataset
2. Sample a random timestep $t \sim \text{Uniform}\{1, \dots, T\}$
3. Sample noise $\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$
4. Compute $\mathbf{x}_t$ in one shot using the closed-form formula
5. Predict $\hat{\boldsymbol{\varepsilon}} = \boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, t)$
6. Minimize $\|\boldsymbol{\varepsilon} - \hat{\boldsymbol{\varepsilon}}\|^2$

In [ ]:
#@title TensorBoard

%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
#@title Diffusion schedule + EMA + checkpoint helpers

class DiffusionSchedule:
    """Holds all precomputed noise-schedule tensors on `device`."""
    def __init__(self, betas, device):
        self.T = len(betas)
        self.betas = betas.to(device)
        self.alphas = (1.0 - self.betas).to(device)
        self.alpha_bar = torch.cumprod(self.alphas, 0).to(device)
        self.sqrt_alpha_bar = self.alpha_bar.sqrt()
        self.sqrt_one_minus_alpha_bar = (1.0 - self.alpha_bar).sqrt()
        self.sqrt_recip_alpha = (1.0 / self.alphas).sqrt()
        self.beta_over_sqrt_one_minus = self.betas / self.sqrt_one_minus_alpha_bar
        alpha_bar_prev = F.pad(self.alpha_bar[:-1], (1, 0), value=1.0)
        self.posterior_var = self.betas * (1.0 - alpha_bar_prev) / (1.0 - self.alpha_bar)


class EMA:
    """Exponential Moving Average of model parameters."""
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.clone() for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k].lerp_(v, 1 - self.decay)

    def apply(self, model):
        self.backup = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)

    def restore(self, model):
        model.load_state_dict(self.backup)


def save_checkpoint(name, model, optimizer, history, ema=None):
    data = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "history": history}
    if ema is not None:
        data["ema"] = ema.shadow
    torch.save(data, CHECKPOINT_DIR / f"{name}.pt")

def load_checkpoint(name, model, optimizer, ema=None):
    path = CHECKPOINT_DIR / f"{name}.pt"
    if path.exists():
        ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        if ema is not None and "ema" in ckpt:
            ema.shadow = ckpt["ema"]
        return ckpt["history"]
    return None


schedule = DiffusionSchedule(betas, DEVICE)

In [ ]:
#@title Train the class-conditional DDPM

optimizer = optim.Adam(model.parameters(), lr=LR)
ema = EMA(model)

# Fixed noise for consistent generation tracking across epochs
fixed_z = torch.randn(NUM_CLASSES, 1, 28, 28, device=DEVICE)
fixed_labels = torch.arange(NUM_CLASSES, device=DEVICE)

@torch.no_grad()
def generate_for_logging(model, schedule, z, y):
    """Quick DDPM sampling from fixed noise for TensorBoard logging."""
    model.eval()
    x = z.clone()
    for t_idx in reversed(range(schedule.T)):
        t_batch = torch.full((x.shape[0],), t_idx, device=x.device, dtype=torch.long)
        eps_pred = model(x, t_batch, y)
        mean = schedule.sqrt_recip_alpha[t_idx] * (
            x - schedule.beta_over_sqrt_one_minus[t_idx] * eps_pred
        )
        if t_idx > 0:
            sigma = schedule.posterior_var[t_idx].sqrt()
            x = mean + sigma * torch.randn_like(x)
        else:
            x = mean
    return x

history = load_checkpoint("cddpm_unet", model, optimizer, ema)
if history is None:
    history = []
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            batch_size = images.shape[0]

            t = torch.randint(0, T, (batch_size,), device=DEVICE)

            eps = torch.randn_like(images)
            abar_t = schedule.alpha_bar[t].view(-1, 1, 1, 1)
            x_t = abar_t.sqrt() * images + (1 - abar_t).sqrt() * eps

            eps_pred = model(x_t, t, labels)
            loss = F.mse_loss(eps_pred, eps)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            ema.update(model)
            epoch_loss += loss.item() * batch_size

        avg_loss = epoch_loss / len(train_data)
        history.append(avg_loss)

        # TensorBoard: scalar loss
        writer.add_scalar("Loss/mse", avg_loss, epoch)

        # TensorBoard: generate samples every 20 epochs (uses EMA weights)
        if epoch == 0 or (epoch + 1) % 20 == 0:
            ema.apply(model)
            generated = generate_for_logging(model, schedule, fixed_z, fixed_labels)
            # Rescale from [-1, 1] to [0, 1] for TensorBoard
            generated = (generated + 1) / 2
            writer.add_image(
                "Generation/fixed_z_all_classes",
                make_grid(generated, nrow=NUM_CLASSES),
                epoch,
            )
            ema.restore(model)

            console.print(
                f"Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.5f} | Samples logged"
            )

    save_checkpoint("cddpm_unet", model, optimizer, history, ema)

writer.flush()
console.print(f"[bold green]Training complete.[/bold green] Final loss = {history[-1]:.5f}")

In [ ]:
#@title Training loss curve

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, len(history) + 1)), y=history,
    mode="lines+markers",
    line=dict(color="#6366f1", width=2),
    marker=dict(size=5),
    name="MSE Loss",
))
fig.update_layout(
    title="DDPM Training Loss",
    xaxis_title="Epoch",
    yaxis_title="MSE Loss (\u03b5 prediction)",
    height=350, width=700,
    margin=dict(t=50, b=50, l=60, r=20),
    template="plotly_white",
)
fig.show()

## 6. DDPM Sampling

Generate images by running the reverse process: start from pure noise $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and iterate $T$ denoising steps.

In [ ]:
#@title DDPM sampling loop

@torch.no_grad()
def ddpm_sample(model, schedule, shape, y, device, save_every=None):
    """
    Generate class-conditional samples via DDPM reverse process.

    Args:
        shape:      (B, C, H, W)
        y:          (B,) integer class labels
        save_every: if set, store intermediate x_t every N steps

    Returns:
        x_0:           (B, C, H, W) final generated samples
        intermediates: list of (t, x_t) tuples
    """
    model.eval()
    x = torch.randn(shape, device=device)
    intermediates = []

    for t_idx in reversed(range(schedule.T)):
        t_batch = torch.full((shape[0],), t_idx, device=device, dtype=torch.long)
        eps_pred = model(x, t_batch, y)
        mean = schedule.sqrt_recip_alpha[t_idx] * (
            x - schedule.beta_over_sqrt_one_minus[t_idx] * eps_pred
        )
        if t_idx > 0:
            sigma = schedule.posterior_var[t_idx].sqrt()
            z = torch.randn_like(x)
            x = mean + sigma * z
        else:
            x = mean
        if save_every and t_idx % save_every == 0:
            intermediates.append((t_idx, x.cpu().clone()))

    return x, intermediates

In [ ]:
#@title Denoising trajectory (EMA weights, one sample per class)

display_steps = [999, 800, 600, 400, 200, 100, 50, 20, 5, 0]
n_cols = len(display_steps)
n_rows = NUM_CLASSES

ema.apply(model)
torch.manual_seed(0)
y = torch.arange(NUM_CLASSES, device=DEVICE)
x = torch.randn(NUM_CLASSES, 1, 28, 28, device=DEVICE)
snaps = {}
model.eval()
with torch.no_grad():
    for t_idx in reversed(range(schedule.T)):
        t_batch = torch.full((NUM_CLASSES,), t_idx, device=DEVICE, dtype=torch.long)
        eps_pred = model(x, t_batch, y)
        mean = schedule.sqrt_recip_alpha[t_idx] * (
            x - schedule.beta_over_sqrt_one_minus[t_idx] * eps_pred)
        if t_idx > 0:
            sigma = schedule.posterior_var[t_idx].sqrt()
            x = mean + sigma * torch.randn_like(x)
        else:
            x = mean
        if t_idx in display_steps:
            snaps[t_idx] = x.cpu().clone()
ema.restore(model)

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f"t={t}" for t in display_steps] + [""] * (n_cols * (n_rows - 1)),
    row_titles=CLASS_NAMES,
    horizontal_spacing=0.01, vertical_spacing=0.02,
)
for row in range(n_rows):
    for col, t in enumerate(display_steps, 1):
        img = snaps[t][row].squeeze().numpy()
        fig.add_trace(
            go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False,
                       hovertemplate="(%{x}, %{y}): %{z:.2f}<extra></extra>"),
            row=row + 1, col=col,
        )
for r in range(1, n_rows + 1):
    for c in range(1, n_cols + 1):
        fig.update_xaxes(showticklabels=False, row=r, col=c)
        fig.update_yaxes(showticklabels=False, row=r, col=c)
fig.update_layout(
    title_text="Denoising Trajectory (one sample per class)",
    height=n_rows * 80 + 80, width=900,
    margin=dict(t=60, b=10, l=40, r=10),
)
fig.show()

In [ ]:
#@title Class-conditional samples \u2014 8 per class

N_PER_CLASS = 8

ema.apply(model)
fig = make_subplots(
    rows=NUM_CLASSES, cols=N_PER_CLASS,
    vertical_spacing=0.01, horizontal_spacing=0.01,
    row_titles=CLASS_NAMES,
)

for c in range(NUM_CLASSES):
    labels = torch.full((N_PER_CLASS,), c, dtype=torch.long, device=DEVICE)
    samples, _ = ddpm_sample(
        model, schedule, (N_PER_CLASS, 1, 28, 28), labels, DEVICE
    )
    samples = samples.cpu()
    for i in range(N_PER_CLASS):
        img = samples[i].squeeze().numpy()
        fig.add_trace(
            go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False,
                       hovertemplate="(%{x}, %{y}): %{z:.2f}<extra></extra>"),
            row=c + 1, col=i + 1,
        )

for r in range(1, NUM_CLASSES + 1):
    for c in range(1, N_PER_CLASS + 1):
        fig.update_xaxes(showticklabels=False, row=r, col=c)
        fig.update_yaxes(showticklabels=False, row=r, col=c)
ema.restore(model)

fig.update_layout(
    title_text="Class-Conditional DDPM \u2014 8 samples per class",
    height=NUM_CLASSES * 100 + 60, width=900,
    margin=dict(t=50, b=10, l=100, r=10),
)
fig.show()

## 7. Interactive Class-Conditional Generation

Pick a class and generate samples. Use the **Resample** button to draw new starting noise.

In [ ]:
#@title Interactive generation

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
from IPython.display import display

ema.apply(model)
model.eval()
N_GEN = 8

class_dropdown = widgets.Dropdown(
    options={CLASS_NAMES[i]: i for i in range(NUM_CLASSES)}, value=7, description="Class:",
)
resample_btn = widgets.Button(description="Resample", button_style="info", icon="random")
progress_label = widgets.Label(value="")


def generate_samples(class_idx):
    """Run full DDPM sampling for the given class."""
    labels = torch.full((N_GEN,), class_idx, dtype=torch.long, device=DEVICE)
    samples, _ = ddpm_sample(model, schedule, (N_GEN, 1, 28, 28), labels, DEVICE)
    return samples.cpu()


def make_gen_grid(samples):
    """Concatenate samples into a single row image."""
    return np.concatenate([samples[i].squeeze().numpy() for i in range(N_GEN)], axis=1)


progress_label.value = "Generating..."
init_samples = generate_samples(7)
progress_label.value = ""

fig_gen = go.FigureWidget(
    data=[go.Heatmap(z=make_gen_grid(init_samples)[::-1], colorscale="Gray_r", showscale=False)],
    layout=dict(title=f"Generated: {CLASS_NAMES[7]}", height=200, width=700,
                xaxis=dict(showticklabels=False, scaleanchor="y"),
                yaxis=dict(showticklabels=False),
                margin=dict(t=40, b=10, l=10, r=10)),
)


def render(change=None):
    progress_label.value = "Generating..."
    samples = generate_samples(class_dropdown.value)
    grid = make_gen_grid(samples)
    with fig_gen.batch_update():
        fig_gen.data[0].z = grid[::-1]
        fig_gen.layout.title.text = f"Generated: {CLASS_NAMES[class_dropdown.value]}"
    progress_label.value = ""


def on_resample(btn):
    render()


class_dropdown.observe(render, names="value")
resample_btn.on_click(on_resample)

display(widgets.VBox([widgets.HBox([class_dropdown, resample_btn, progress_label]), fig_gen]))

## 8. Interactive Denoising Viewer

Watch the reverse process unfold for a single class. Use the slider to scrub through timesteps and see how the image emerges from noise.

In [ ]:
#@title Interactive denoising viewer

model.eval()

# Pre-generate a full denoising trajectory for one class
viewer_class = widgets.Dropdown(
    options={CLASS_NAMES[i]: i for i in range(NUM_CLASSES)}, value=7, description="Class:",
)
viewer_btn = widgets.Button(description="New trajectory", button_style="info", icon="play")
viewer_progress = widgets.Label(value="")

# Store snapshots at regular intervals
SNAP_EVERY = 10  # save every 10 steps
N_VIEWER = 4  # number of parallel samples


def run_trajectory(class_idx):
    """Run full denoising and save snapshots every SNAP_EVERY steps."""
    labels = torch.full((N_VIEWER,), class_idx, dtype=torch.long, device=DEVICE)
    x = torch.randn(N_VIEWER, 1, 28, 28, device=DEVICE)
    snaps = {}
    with torch.no_grad():
        for t_idx in reversed(range(schedule.T)):
            t_batch = torch.full((N_VIEWER,), t_idx, device=DEVICE, dtype=torch.long)
            eps_pred = model(x, t_batch, labels)
            mean = schedule.sqrt_recip_alpha[t_idx] * (
                x - schedule.beta_over_sqrt_one_minus[t_idx] * eps_pred)
            if t_idx > 0:
                sigma = schedule.posterior_var[t_idx].sqrt()
                x = mean + sigma * torch.randn_like(x)
            else:
                x = mean
            if t_idx % SNAP_EVERY == 0:
                snaps[t_idx] = x.cpu().clone()
    return snaps


def make_viewer_grid(snaps, t_idx):
    """Create a row of images from the snapshot at timestep t_idx."""
    # Find the closest available snapshot
    available = sorted(snaps.keys())
    closest = min(available, key=lambda k: abs(k - t_idx))
    imgs = snaps[closest]
    return np.concatenate([imgs[i].squeeze().numpy() for i in range(N_VIEWER)], axis=1)


viewer_progress.value = "Running trajectory..."
viewer_snaps = run_trajectory(7)
viewer_progress.value = ""

t_slider = widgets.IntSlider(
    value=0, min=0, max=T - 1, step=SNAP_EVERY,
    description="Timestep:",
    continuous_update=True,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="600px"),
)

fig_viewer = go.FigureWidget(
    data=[go.Heatmap(z=make_viewer_grid(viewer_snaps, 0)[::-1],
                     colorscale="Gray_r", showscale=False)],
    layout=dict(title=f"t = 0 | {CLASS_NAMES[7]}", height=200, width=500,
                xaxis=dict(showticklabels=False, scaleanchor="y"),
                yaxis=dict(showticklabels=False),
                margin=dict(t=40, b=10, l=10, r=10)),
)


def update_viewer(change=None):
    t_val = t_slider.value
    grid = make_viewer_grid(viewer_snaps, t_val)
    with fig_viewer.batch_update():
        fig_viewer.data[0].z = grid[::-1]
        fig_viewer.layout.title.text = f"t = {t_val} | {CLASS_NAMES[viewer_class.value]}"


def new_trajectory(btn):
    global viewer_snaps
    viewer_progress.value = "Running trajectory..."
    viewer_snaps = run_trajectory(viewer_class.value)
    viewer_progress.value = ""
    t_slider.value = 0
    update_viewer()


t_slider.observe(update_viewer, names="value")
viewer_btn.on_click(new_trajectory)

display(widgets.VBox([
    widgets.HBox([viewer_class, viewer_btn, viewer_progress]),
    t_slider,
    fig_viewer,
]))

## 9. Interactive Noise Schedule Explorer

Adjust the noise schedule parameters and see how they change the forward noising process on a real image.

In [ ]:
#@title Interactive schedule explorer

schedule_type = widgets.Dropdown(
    options=["Linear", "Cosine"], value="Cosine", description="Schedule:",
)
beta_start_slider = widgets.FloatLogSlider(
    value=1e-4, base=10, min=-5, max=-2, step=0.1,
    description="\u03b2_start:", readout_format=".1e",
    layout=widgets.Layout(width="500px"),
)
beta_end_slider = widgets.FloatLogSlider(
    value=0.02, base=10, min=-3, max=-0.5, step=0.1,
    description="\u03b2_end:", readout_format=".1e",
    layout=widgets.Layout(width="500px"),
)
cosine_s_slider = widgets.FloatSlider(
    value=0.008, min=0.001, max=0.1, step=0.001,
    description="s (cosine):", readout_format=".3f",
    layout=widgets.Layout(width="500px"),
)
timestep_slider = widgets.IntSlider(
    value=500, min=0, max=T - 1, step=10,
    description="Timestep:",
    layout=widgets.Layout(width="500px"),
)

# Use the same sample image from earlier
x_0_explore = sample_img.unsqueeze(0)
torch.manual_seed(123)
fixed_noise = torch.randn_like(x_0_explore)

# Two subplots: schedule curve + noised image
fig_explore = go.FigureWidget(
    make_subplots(rows=1, cols=2,
                  subplot_titles=["\u0101\u0332_t curve", "Noised image"],
                  column_widths=[0.6, 0.4]),
)
fig_explore.add_trace(
    go.Scatter(x=list(range(T)), y=alpha_bar_cosine.numpy().tolist(),
               line=dict(color="#4A90D9", width=2), name="\u0101\u0332_t"),
    row=1, col=1,
)
fig_explore.add_trace(
    go.Scatter(x=[500], y=[float(alpha_bar_cosine[500])],
               mode="markers", marker=dict(size=12, color="#E74C3C"),
               name="Current t"),
    row=1, col=1,
)
# Noised image
abar_val = float(alpha_bar_cosine[500])
x_t_init = (abar_val ** 0.5) * x_0_explore + ((1 - abar_val) ** 0.5) * fixed_noise
fig_explore.add_trace(
    go.Heatmap(z=x_t_init.squeeze().numpy()[::-1], colorscale="Gray_r", showscale=False),
    row=1, col=2,
)
fig_explore.update_xaxes(showticklabels=False, row=1, col=2)
fig_explore.update_yaxes(showticklabels=False, row=1, col=2)
fig_explore.update_layout(height=300, width=700, margin=dict(t=40, b=40, l=50, r=10),
                          showlegend=False)


def update_explore(change=None):
    stype = schedule_type.value
    if stype == "Linear":
        b = linear_beta_schedule(T, beta_start_slider.value, beta_end_slider.value)
    else:
        b = cosine_beta_schedule(T, s=cosine_s_slider.value)
    abar = torch.cumprod(1.0 - b, dim=0)
    t_val = timestep_slider.value
    abar_t = float(abar[t_val])

    x_t = (abar_t ** 0.5) * x_0_explore + ((1 - abar_t) ** 0.5) * fixed_noise

    with fig_explore.batch_update():
        fig_explore.data[0].y = abar.numpy().tolist()
        fig_explore.data[1].x = [t_val]
        fig_explore.data[1].y = [abar_t]
        fig_explore.data[2].z = x_t.squeeze().numpy()[::-1]


for w in [schedule_type, beta_start_slider, beta_end_slider, cosine_s_slider, timestep_slider]:
    w.observe(update_explore, names="value")

# Show/hide linear params based on schedule type
linear_params = widgets.VBox([beta_start_slider, beta_end_slider])
cosine_params = widgets.VBox([cosine_s_slider])

def toggle_params(change=None):
    if schedule_type.value == "Linear":
        linear_params.layout.display = ""
        cosine_params.layout.display = "none"
    else:
        linear_params.layout.display = "none"
        cosine_params.layout.display = ""
    update_explore()

schedule_type.observe(toggle_params, names="value")
toggle_params()  # initial state

display(widgets.VBox([
    schedule_type,
    linear_params,
    cosine_params,
    timestep_slider,
    fig_explore,
]))

## 10. Interactive Multi-Class Comparison

Select multiple classes and generate samples side-by-side for direct comparison.

In [ ]:
#@title Interactive multi-class comparison

model.eval()

class_select = widgets.SelectMultiple(
    options={CLASS_NAMES[i]: i for i in range(NUM_CLASSES)},
    value=[7, 9, 1],
    description="Classes:",
    rows=5,
)
n_samples_slider = widgets.IntSlider(
    value=6, min=2, max=10, step=1, description="Samples:",
)
compare_btn = widgets.Button(description="Generate", button_style="success", icon="bolt")
compare_progress = widgets.Label(value="")

fig_compare = go.FigureWidget(
    layout=dict(title="Select classes and click Generate", height=100, width=700),
)


def run_comparison(btn):
    selected = list(class_select.value)
    if not selected:
        compare_progress.value = "Select at least one class!"
        return
    n = n_samples_slider.value
    compare_progress.value = f"Generating {len(selected)} classes \u00d7 {n} samples..."

    all_samples = []
    for c in selected:
        labels = torch.full((n,), c, dtype=torch.long, device=DEVICE)
        samps, _ = ddpm_sample(model, schedule, (n, 1, 28, 28), labels, DEVICE)
        all_samples.append(samps.cpu())

    new_fig = make_subplots(
        rows=len(selected), cols=n,
        vertical_spacing=0.02, horizontal_spacing=0.01,
        row_titles=[CLASS_NAMES[c] for c in selected],
    )
    for r, (c, samps) in enumerate(zip(selected, all_samples)):
        for i in range(n):
            img = samps[i].squeeze().numpy()
            new_fig.add_trace(
                go.Heatmap(z=img[::-1], colorscale="Gray_r", showscale=False),
                row=r + 1, col=i + 1,
            )
    for r in range(1, len(selected) + 1):
        for c in range(1, n + 1):
            new_fig.update_xaxes(showticklabels=False, row=r, col=c)
            new_fig.update_yaxes(showticklabels=False, row=r, col=c)
    new_fig.update_layout(
        height=len(selected) * 110 + 40, width=700,
        margin=dict(t=30, b=10, l=100, r=10),
    )

    fig_compare.data = []
    for trace in new_fig.data:
        fig_compare.add_trace(trace)
    fig_compare.layout = new_fig.layout
    compare_progress.value = ""


compare_btn.on_click(run_comparison)

display(widgets.VBox([
    widgets.HBox([class_select, widgets.VBox([n_samples_slider, compare_btn, compare_progress])]),
    fig_compare,
]))

In [ ]:
#@title Restore training weights (cleanup)

ema.restore(model)
console.print("[bold green]EMA weights restored \u2014 back to training weights.[/bold green]")

---

**Full tutorial:** [The Diffusion Paradigm](https://overfitting.club/posts/tutorials/deep_learning/the_diffusion_paradigm/the_diffusion_paradigm.html) on The Overfitting Club.